In [ ]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine,text


In [ ]:

df = pd.read_csv("HR_Employee_Attrition.csv")

In [ ]:


# 3. Initial inspection (rows, columns, info)
print(df.head())


In [ ]:
print(df.info())


In [ ]:
print(df.shape)


In [ ]:

# 4. Standardize/clean column names (strip spaces, lower case)
df.columns = [col.strip().replace(" ", "_").lower() for col in df.columns]


In [ ]:

# 5. Remove duplicate rows
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()


In [ ]:

# 6. Identify missing values & weird NAs
print(df.isnull().sum())
print(df.isin(['NA', 'N/A', ' ', '']).sum())


In [ ]:

# 7. Replace missing/blank/NA values with np.nan
df.replace(['NA', 'N/A', ' ', ''], np.nan, inplace=True)


In [ ]:

# 8. Impute/fill missing values for each column (numeric: median, categorical: mode)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == "O":
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)


In [ ]:

# 9. Convert object columns to numeric IF possible (auto-detect)
for col in df.columns:
    if df[col].dtype == "object":
        try:
            df[col] = pd.to_numeric(df[col], errors="raise")
            print(f"{col} converted to numeric.")
        except:
            pass  # Not numeric, leave as object


In [ ]:

# 10. Remove irrelevant columns with single unique value (waste for analysis)
for col in ['employeecount', 'standardhours', 'over18']:
    if col in df.columns and df[col].nunique() == 1:
        df.drop(col, axis=1, inplace=True)
        print(f"{col} dropped.")


In [ ]:

# 11. Strip trailing spaces from strings (object columns)
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()


In [ ]:

# 12. Double check types and uniques (for plotting/groupby)
print(df.info())
print("Unique value count per column:\n", df.nunique())


In [ ]:

# 13. Categorical value_counts for EDA
for col in df.select_dtypes(include="object").columns:
    print(f"\nValue counts for {col}:\n", df[col].value_counts())


In [ ]:

# 14. Export cleaned data for EDA/analysis
df.to_csv("hr_attrition_cleaned_data.csv", index=False)
print("Cleaned file saved as 'hr_attrition_cleaned_data.csv'.")


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
import pandas as pd
from sqlalchemy import create_engine


In [ ]:

# 1. Load your cleaned dataset
df_clean= pd.read_csv("hr_attrition_cleaned_data.csv")


In [ ]:

# 2. Create connection engine to MySQL
username = "root"    
password = "pricass00"    
host = "localhost"
database = "hr_analytics"           

connection_string = f"mysql+pymysql://{username}:{password}@{host}/{database}"
engine = create_engine(connection_string)


In [ ]:

# 3. Upload DataFrame to a MySQL table called 'attrition'; overwrite if exists
df_clean.to_sql("employee_attrition", engine, if_exists="replace", index=False)
print("Cleaned dataset loaded to MySQL table 'attrition' successfully!")


In [ ]:
df_clean.shape

In [ ]:
df_clean.info()

In [ ]:
df_sql = pd.read_sql("SELECT * FROM employee_attrition", con=engine)


In [ ]:
df_sql.shape

In [ ]:

df_sql.head()


In [ ]:
 
print(df_sql.info())     